# Setup

Before start, please make sure you have all the dependencies installed.

In [ ]:
import numba
import numcodecs
import numpy as np
import zarr
import zarr.storage
from zarr.abc.store import ByteRequest
from zarr.core.buffer import Buffer, BufferPrototype

## Zoomable Mandelbrot Set

This notebook a contains `vizarr` example visualizing a generic multiscale zarr. The first cell contains code to create the underlying generatvie zarr store. It dynamically creates "chunks" at different zoom levels and associated array metadata.

In [ ]:

@numba.njit
def mandelbrot(out, from_x, from_y, to_x, to_y, grid_size, maxiter):  # noqa: ANN001, ANN201, D103, PLR0913
    step_x = (to_x - from_x) / grid_size
    step_y = (to_y - from_y) / grid_size
    creal = from_x
    cimag = from_y
    for i in range(grid_size):
        cimag = from_y
        for j in range(grid_size):
            nreal = real = imag = n = 0
            for _ in range(maxiter):
                nreal = real * real - imag * imag + creal
                imag = 2 * real * imag + cimag
                real = nreal
                if real * real + imag * imag > 4.0:  # noqa: PLR2004
                    break
                n += 1
            out[j * grid_size + i] = n
            cimag += step_y
        creal += step_x
    return out


@numba.njit
def tile_bounds(level, x, y, max_level, min_coord=-2.5, max_coord=2.5):  # noqa: ANN001, ANN201, D103, PLR0913
    max_width = max_coord - min_coord
    tile_width = max_width / 2 ** (max_level - level)
    from_x = min_coord + x * tile_width
    to_x = min_coord + (x + 1) * tile_width
    from_y = min_coord + y * tile_width
    to_y = min_coord + (y + 1) * tile_width
    return from_x, from_y, to_x, to_y


class MandlebrotStore(zarr.storage.WrapperStore[zarr.storage.MemoryStore]):  # noqa: D101

    def __init__(
        self,
        levels: int,
        tilesize: int,
        maxiter: int = 255,
        compressor: numcodecs.abc.Codec | None = None,
    ) -> None:
        # Build metadata using zarr's own API
        meta = zarr.storage.MemoryStore()
        super().__init__(meta)
        self.levels = levels
        self.tilesize = tilesize
        self.compressor = compressor
        self.dtype = np.dtype(np.uint8 if maxiter < 256 else np.uint16)  # noqa: PLR2004
        self.maxiter = maxiter

        grp = zarr.create_group(meta, zarr_format=2)
        base_width = tilesize * 2**levels
        for level in range(levels):
            width = int(base_width / 2**level)
            zarr.create_array(
                meta,
                name=str(level),
                shape=(width, width),
                chunks=(tilesize, tilesize),
                dtype=self.dtype,
                compressors=compressor,
                zarr_format=2,
            )
        datasets = [{"path": str(i)} for i in range(levels)]
        multiscales = [{"datasets": datasets, "version": "0.1"}]
        grp.update_attributes({"multiscales": multiscales})

    def _compute_chunk(self, key: str) -> bytes | None:
        try:
            level_str, chunk_key = key.split("/")
            level = int(level_str)
            y, x = map(int, chunk_key.split("."))
        except (ValueError, AttributeError):
            return None

        from_x, from_y, to_x, to_y = tile_bounds(
            level, x, y, self.levels,
        )
        out = np.zeros(
            self.tilesize * self.tilesize, dtype=self.dtype,
        )
        tile = mandelbrot(
            out, from_x, from_y, to_x, to_y,
            self.tilesize, self.maxiter,
        )
        tile = tile.reshape(self.tilesize, self.tilesize)
        if self.compressor:
            return self.compressor.encode(tile)
        return tile.tobytes()

    async def get(
        self,
        key: str,
        prototype: BufferPrototype,
        byte_range: ByteRequest | None = None,
    ) -> Buffer | None:
        """Get metadata or compute a chunk on the fly."""
        buf = await super().get(key, prototype, byte_range)
        if buf is not None:
            return buf
        chunk = self._compute_chunk(key)
        if chunk is not None:
            return prototype.buffer.from_bytes(chunk)
        return None

    async def exists(self, key: str) -> bool:  # noqa: D102
        if await super().exists(key):
            return True
        return self._compute_chunk(key) is not None

### Running vizarr

Initalize the store implemented above, and open as a `zarr.Group` for vizarr. 

In [ ]:
# Initialize the store
store = MandlebrotStore(levels=50, tilesize=512, compressor=numcodecs.Blosc())

In [ ]:
import vizarr

viewer = vizarr.Viewer()
viewer.add_image(source=store, name="mandelbrot")
viewer